# Visual Semantic Encoder

This notebook keeps the project training flow in one place. The imports and COCO dataset setup follow the shared notebook layout, while the custom CNN architecture remains the project implementation.


## Configuration

Set the dataset location, output folder, and main training options here. On Kaggle, the notebook uses the mounted COCO dataset and working directory automatically. Locally, replace `DATASET_PATH` with the folder containing `train2017/`, `val2017/`, and `annotations/`.


In [ ]:
# Choose the dataset and output locations for Kaggle or local use.
import os, torch

ON_KAGGLE = os.path.exists("/kaggle")

if ON_KAGGLE:
    DATASET_PATH = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017"
    OUTPUT_DIR = "/kaggle/working/"
else:
    DATASET_PATH = r"D:\path\to\coco2017"  # change this for local training
    OUTPUT_DIR = "checkpoints_custom"

# Main training settings. Adjust these before starting a run.
NUM_EPOCHS = 10
SUBSET_SIZE = 100000
BATCH_SIZE = 64
LR = 4e-4
FREQ_THRESHOLD = 5
EMBED_SIZE, HIDDEN_SIZE, ATTENTION_DIM = 256, 512, 256

RESUME_FROM = None  # e.g. "checkpoints_custom/checkpoint_img_caption.pth"
NUM_WORKERS = 2 if ON_KAGGLE else 0

# Create the output folder and define the files written by training.
os.makedirs(OUTPUT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "checkpoint_img_caption.pth")
VOCAB_PATH = os.path.join(OUTPUT_DIR, "vocab.txt")
print("Outputs will be saved to:", os.path.abspath(OUTPUT_DIR))


## Shared imports

These imports cover file paths, image preprocessing, plotting, progress display, and the PyTorch components used by the dataset, model, and training loop.


In [ ]:
# Standard library helpers for data loading and reproducibility.
import json, math, random, re, sys, warnings
from collections import Counter
from functools import partial

# Image and plotting utilities used by the notebook.
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

# PyTorch modules used by the dataset, model, and optimizer.
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as transforms

# Keep data-order and weight initialization choices repeatable.
random.seed(0)
torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Vocabulary

The vocabulary converts caption words into integer IDs. It also stores the special padding, start, end, and unknown tokens used by the decoder.


In [ ]:
# Map caption words to IDs and preserve the special sequence tokens.
class Vocabulary:
    def __init__(self, freqthreshold=2):
        self.freq_threshold = freqthreshold
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {v: k for k, v in self.itos.items()}

    def __len__(self):
        return len(self.itos)

    @staticmethod
    def tokenizer(text):
        return re.findall(r"\b\w+\b", text.lower())

    # Count words in the training captions and assign IDs to frequent words.
    def buildvocabulary(self, sentence_list):
        frequencies = Counter()
        idx = 4
        for sentence in sentence_list:
            frequencies.update(self.tokenizer(sentence))
        for word, freq in frequencies.items():
            if freq >= self.freq_threshold:
                self.stoi[word] = idx
                self.itos[idx] = word
                idx += 1

    def numericalize(self, text):
        return [self.stoi.get(token, self.stoi["<UNK>"]) for token in self.tokenizer(text)]

    def save(self, path):
        """Write the vocabulary as one 'index:word' line per entry."""
        with open(path, "w", encoding="utf-8") as f:
            for idx in sorted(self.itos):
                f.write(f"{idx}:{self.itos[idx]}\n")

    @classmethod
    def from_itos(cls, itos):
        vocab = cls(freqthreshold=1)
        vocab.itos = {int(k): w for k, w in itos.items()}
        vocab.stoi = {w: k for k, w in vocab.itos.items()}
        return vocab

    @classmethod
    def load(cls, path):
        """Read a vocabulary written by save()."""
        with open(path, "r", encoding="utf-8") as f:
            entries = [line.rstrip("\n").split(":", 1) for line in f if line.strip()]
        return cls.from_itos({int(k): w for k, w in entries})


## COCO dataset setup

This section reads COCO caption annotations, pairs each caption with its image, applies the same 224x224 normalization used by the model, and batches padded captions for training.


In [ ]:
# Read COCO annotations and pair each caption with its image path.
def load_coco_pairs(dataset_path, split):
    """Return [(image_path, caption), ...] for a COCO split."""
    json_path = os.path.join(dataset_path, "annotations", f"captions_{split}.json")
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    id_to_file = {img["id"]: img["file_name"] for img in data["images"]}
    return [
        (os.path.join(dataset_path, split, id_to_file[ann["image_id"]]), ann["caption"])
        for ann in data["annotations"]
        if ann["image_id"] in id_to_file
    ]


# Convert each image and caption into tensors for the DataLoader.
class CocoDataset(Dataset):
    def __init__(self, data_pairs, vocab, transform=None):
        self.data_pairs, self.vocab, self.transform = data_pairs, vocab, transform

    def __len__(self):
        return len(self.data_pairs)

    def __getitem__(self, idx):
        imgpath, caption = self.data_pairs[idx]
        image = Image.open(imgpath).convert("RGB")
        if self.transform:
            image = self.transform(image)
        ids = [self.vocab.stoi["<SOS>"]] + self.vocab.numericalize(caption) + [self.vocab.stoi["<EOS>"]]
        return image, torch.tensor(ids)


# Stack images and pad captions to a common length within each batch.
def collate_fn(batch, pad_idx=0):
    images, captions = zip(*batch)
    images = torch.stack(images, 0)
    captions = pad_sequence(list(captions), batch_first=True, padding_value=pad_idx)
    return images, captions


# Apply the preprocessing expected by the CNN.
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Build the vocabulary from the training captions, then create the loader.
train_pairs = load_coco_pairs(DATASET_PATH, "train2017")
print("Total training captions:", len(train_pairs))

vocab = Vocabulary(freqthreshold=FREQ_THRESHOLD)
vocab.buildvocabulary([caption for _, caption in train_pairs])
vocab.save(VOCAB_PATH)
print("Vocabulary size:", len(vocab), "-> saved to", VOCAB_PATH)

dataset = CocoDataset(train_pairs, vocab, transform=eval_transform)
n_use = min(SUBSET_SIZE or len(dataset), len(dataset))
dataloader = DataLoader(
    Subset(dataset, list(range(n_use))),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=partial(collate_fn, pad_idx=vocab.stoi["<PAD>"]),
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)
print(f"Training on {n_use} captions, {len(dataloader)} batches/epoch")


## Attention

Additive attention scores every image region using both the encoded image features and the decoder hidden state. It then creates one weighted context vector for the current word.


In [ ]:
# Project image features and decoder state into a shared attention space.
class Attention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super(Attention, self).__init__()
        self.encoder_att = nn.Linear(encoder_dim, attention_dim)
        self.decoder_att = nn.Linear(decoder_dim, attention_dim)
        self.full_att = nn.Linear(attention_dim, 1)
    
    # Produce attention weights and the weighted image context.
    def forward(self, encoder_out, hidden_state):
        """
        encoder_out: (B, num_pixels, encoder_dim)
        hidden_state: (B, decoder_dim)
        """
        att1 = self.encoder_att(encoder_out)       # (B, num_pixels, att_dim)
        att2 = self.decoder_att(hidden_state).unsqueeze(1)  # (B, 1, att_dim)
        att = torch.tanh(att1 + att2)              # (B, num_pixels, att_dim)
        alpha = F.softmax(self.full_att(att), dim=1)  # (B, num_pixels, 1)
        context = (encoder_out * alpha).sum(dim=1)    # (B, encoder_dim)
        return context, alpha


## Decoder

The decoder embeds the input caption, combines each word embedding with the attention context, and updates an LSTMCell one time step at a time.


In [ ]:
# Generate one vocabulary prediction for each caption time step.
class DecoderWithAttention(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, attention_dim=256, dropout=0.3):
        super(DecoderWithAttention, self).__init__()
        self.attention = Attention(embed_size, hidden_size, attention_dim)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTMCell(embed_size + embed_size, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(dropout)
    
    # Use teacher forcing: the decoder receives all caption inputs at once.
    def forward(self, encoder_out, captions):
        """
        encoder_out: (B, num_pixels, embed_size)
        captions: (B, T)
        """
        batch_size = encoder_out.size(0)
        embeddings = self.embedding(captions[:, :-1])  # (B, T-1, embed)
        h = torch.zeros(batch_size, self.lstm.hidden_size, device=encoder_out.device)
        c = torch.zeros(batch_size, self.lstm.hidden_size, device=encoder_out.device)
        outputs = []

        for t in range(embeddings.size(1)):
            context, _ = self.attention(encoder_out, h)  # (B, embed_size)
            lstm_input = torch.cat([embeddings[:, t, :], context], dim=1)  # (B, 2*embed)
            h, c = self.lstm(lstm_input, (h, c))
            out = self.fc(self.dropout(h))  # (B, vocab_size)
            outputs.append(out.unsqueeze(1))
        
        outputs = torch.cat(outputs, dim=1)  # (B, T-1, vocab_size)
        return outputs


## Encoder

The encoder is the custom ResNet-50-style CNN. It turns each image into a 7x7 grid of 2048-channel features and projects each of the 49 regions into the decoder embedding size.


In [ ]:
# Each bottleneck reduces, processes, and expands channels before the skip connection.
class Bottleneck(nn.Module):
    """1x1 reduce -> 3x3 -> 1x1 expand, with a skip connection."""
    expansion = 4

    def __init__(self, in_channels, mid_channels, stride=1):
        super().__init__()
        out_channels = mid_channels * self.expansion
        self.conv1 = nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_channels)
        self.conv2 = nn.Conv2d(mid_channels, mid_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(mid_channels)
        self.conv3 = nn.Conv2d(mid_channels, out_channels, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        return self.relu(out + identity)


# Assemble the 3-4-6-3 bottleneck stages used by the encoder.
def make_stage(in_channels, mid_channels, num_blocks, stride):
    blocks = [Bottleneck(in_channels, mid_channels, stride)]
    in_channels = mid_channels * Bottleneck.expansion
    for _ in range(num_blocks - 1):
        blocks.append(Bottleneck(in_channels, mid_channels))
    return nn.Sequential(*blocks)


# Convert an image into a spatial grid of trainable visual features.
class EncoderCNN(nn.Module):
    """Custom CNN (ResNet-50-style, trained from scratch, fully trainable).

    (B, 3, 224, 224) -> (B, 2048, 7, 7) -> (B, 49, 2048) -> (B, 49, embed_size)
    """

    def __init__(self, embed_size, dropout_p=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            make_stage(64, 64, 3, stride=1),
            make_stage(256, 128, 4, stride=2),
            make_stage(512, 256, 6, stride=2),
            make_stage(1024, 512, 3, stride=2),
        )
        self.dropout = nn.Dropout(dropout_p)
        self.fc = nn.Linear(2048, embed_size)
        self._init_weights()

    # Initialize the custom CNN instead of loading external weights.
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
        for m in self.modules():
            if isinstance(m, Bottleneck):
                nn.init.zeros_(m.bn3.weight)

    # Flatten the 7x7 feature map into 49 regions for attention.
    def forward(self, images):
        features = self.cnn(images)
        B, C, H, W = features.shape
        features = features.permute(0, 2, 3, 1).reshape(B, H * W, C)
        return self.fc(self.dropout(features))


## Combined model

The combined model connects the encoder to the attention-based decoder and returns vocabulary logits for each caption time step.


In [ ]:
# Connect the visual encoder and the caption decoder.
class CombinedModel(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, attention_dim=256):
        super(CombinedModel, self).__init__()
        self.encoder = EncoderCNN(embed_size)
        self.decoder = DecoderWithAttention(embed_size, hidden_size, vocab_size, attention_dim)

    def forward(self, images, captions):
        encoder_out = self.encoder(images)  # (B, num_pixels, embed)
        outputs = self.decoder(encoder_out, captions)  # (B, T-1, vocab_size)
        return outputs


## Checkpoint helpers

These helpers save model, optimizer, loss, and vocabulary state so training can resume and inference can rebuild the model later.


In [ ]:
# Save enough state to resume training or rebuild the model for inference.
def save_checkpoint(model, optimizer, epoch, loss, path, vocab=None):
    """Save a training checkpoint. If a vocabulary is given it is stored inside the file too."""
    path = str(path)
    folder = os.path.dirname(path)
    if folder:
        os.makedirs(folder, exist_ok=True)
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": loss,
    }
    if vocab is not None:
        payload["vocab_itos"] = dict(vocab.itos)
    tmp_path = path + ".tmp"  # write to a temp file first so a crash cannot corrupt the real checkpoint
    torch.save(payload, tmp_path)
    os.replace(tmp_path, path)


# Restore model and, when available, optimizer state for continued training.
def load_checkpoint(model, optimizer=None, path="checkpoint.pth"):
    """Load weights (and, if possible, the optimizer state) to resume training."""
    checkpoint = torch.load(path, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"])
    if optimizer is not None:
        try:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        except ValueError:
            warnings.warn("Optimizer state could not be restored (parameter groups differ); "
                          "continuing with a fresh optimizer.")
    return checkpoint["epoch"], checkpoint["loss"]


# Recreate model dimensions from the saved checkpoint for inference.
def load_for_inference(checkpoint_path, vocab_path=None, device="cpu"):
    """Rebuild the model and vocabulary from a checkpoint.

    Model sizes are read from the saved weights. The vocabulary comes from the checkpoint if it
    contains one, otherwise from vocab_path (the vocab.txt written during training).
    """
    checkpoint = torch.load(str(checkpoint_path), map_location=device)
    state = checkpoint["model_state_dict"]
    vocab_size = state["decoder.fc.weight"].shape[0]
    embed_size = state["decoder.embedding.weight"].shape[1]
    hidden_size = state["decoder.lstm.weight_hh"].shape[1]
    attention_dim = state["decoder.attention.encoder_att.weight"].shape[0]

    if "vocab_itos" in checkpoint:
        vocab = Vocabulary.from_itos(checkpoint["vocab_itos"])
    elif vocab_path is not None:
        vocab = Vocabulary.load(vocab_path)
    else:
        raise ValueError("This checkpoint has no vocabulary inside it. Pass --vocab path/to/vocab.txt")
    if len(vocab) != vocab_size:
        raise ValueError(f"The vocabulary has {len(vocab)} words but the model expects {vocab_size}. "
                         "Use the vocab.txt that was created together with this checkpoint.")

    model = CombinedModel(embed_size, hidden_size, vocab_size, attention_dim).to(device)
    model.load_state_dict(state)
    model.eval()
    return model, vocab


## Training loop

The training loop performs teacher-forced caption prediction, computes cross-entropy loss while ignoring padding tokens, clips gradients, and saves a checkpoint after each epoch.


In [ ]:
matplotlib.use("Agg")  # save the loss plot to a file; a window would block the script


EMBED_SIZE, HIDDEN_SIZE, ATTENTION_DIM = 256, 512, 256


# Run epochs of teacher-forced training and save progress after each epoch.
def train_model(model, dataloader, criterion, optimizer, vocab, device, num_epochs,
                start_epoch=0, checkpoint_path=None, plot_path=None):
    model.to(device)
    train_losses = []

    for epoch in range(start_epoch, num_epochs):
        model.train()
        total_loss = 0.0

        loop = tqdm(dataloader, desc=f"Epoch [{epoch + 1}/{num_epochs}]", leave=False)
        for images, captions in loop:
            images = images.to(device)
            captions = captions.to(device)

            # The decoder shifts the captions itself (its inputs are captions[:, :-1]),
            # so pass the full captions here and compare against captions[:, 1:].
            outputs = model(images, captions)
            targets = captions[:, 1:]
            loss = criterion(outputs.reshape(-1, outputs.size(-1)), targets.reshape(-1))

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            total_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        avg_loss = total_loss / max(len(dataloader), 1)
        train_losses.append(avg_loss)
        print(f"Epoch [{epoch + 1}/{num_epochs}] Avg Loss: {avg_loss:.4f}")

        if checkpoint_path is not None:
            save_checkpoint(model, optimizer, epoch, avg_loss, checkpoint_path, vocab=vocab)

    if train_losses and plot_path is not None:
        plt.figure(figsize=(8, 5))
        plt.plot(range(start_epoch + 1, start_epoch + len(train_losses) + 1), train_losses, marker="o", color="blue")
        plt.title("Training Loss over Epochs")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.grid(True)
        plt.savefig(plot_path, bbox_inches="tight")
        plt.close()
        print(f"Loss curve saved to {plot_path}")

    return train_losses


# This CLI helper remains available for the original train.py workflow.
def build_vocab_and_dataloader(dataset_dir, batch_size=64, subset_size=100000, freq_threshold=5, num_workers=0):
    data_pairs = load_coco_pairs(dataset_dir, "train2017")

    vocab = Vocabulary(freqthreshold=freq_threshold)
    vocab.buildvocabulary([caption for _, caption in data_pairs])

    dataset = CocoDataset(data_pairs, vocab, transform=build_transform())
    use = min(subset_size, len(dataset)) if subset_size and subset_size > 0 else len(dataset)

    dataloader = DataLoader(
        Subset(dataset, list(range(use))),
        batch_size=batch_size,
        shuffle=True,
        collate_fn=partial(collate_fn, vocab=vocab),
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=num_workers > 0,
    )
    print(f"Training on {use} captions ({len(dataloader)} batches per epoch), vocabulary size {len(vocab)}")
    return vocab, dataloader


# The original command-line entry point is kept for reference.
def main():
    parser = argparse.ArgumentParser(description="Train an image captioning model.")
    parser.add_argument("--dataset-dir", type=str, default="data/coco2017", help="COCO root containing train2017/ and annotations/")
    parser.add_argument("--epochs", type=int, default=10, help="Number of training epochs.")
    parser.add_argument("--batch-size", type=int, default=64, help="Training batch size.")
    parser.add_argument("--subset-size", type=int, default=100000, help="Number of training captions to use. 0 = use all.")
    parser.add_argument("--lr", type=float, default=4e-4, help="Learning rate.")
    parser.add_argument("--freq-threshold", type=int, default=5, help="Minimum word frequency for the vocabulary.")
    parser.add_argument("--num-workers", type=int, default=0 if os.name == "nt" else 2, help="DataLoader worker processes.")
    parser.add_argument("--checkpoint-dir", type=str, default="checkpoints_custom", help="Where checkpoint_img_caption.pth and vocab.txt are written.")
    parser.add_argument("--resume", type=str, default=None, help="Checkpoint to continue training from.")
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)
    checkpoint_dir = Path(args.checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_dir / "checkpoint_img_caption.pth"
    if checkpoint_path.exists() and not args.resume:
        raise FileExistsError(
            f"{checkpoint_path} already exists; pass --resume to continue training instead of overwriting it."
        )

    vocab, dataloader = build_vocab_and_dataloader(
        dataset_dir=args.dataset_dir,
        batch_size=args.batch_size,
        subset_size=args.subset_size,
        freq_threshold=args.freq_threshold,
        num_workers=args.num_workers,
    )
    vocab_path = checkpoint_dir / "vocab.txt"
    vocab.save(vocab_path)
    print(f"Vocabulary saved to {vocab_path}")

    model = CombinedModel(EMBED_SIZE, HIDDEN_SIZE, len(vocab), ATTENTION_DIM).to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi["<PAD>"])
    optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=args.lr)

    start_epoch = 0
    if args.resume:
        last_epoch, last_loss = load_checkpoint(model, optimizer, args.resume)
        start_epoch = last_epoch + 1
        print(f"Resuming after epoch {last_epoch + 1} (loss {last_loss:.4f})")

    train_model(
        model=model,
        dataloader=dataloader,
        criterion=criterion,
        optimizer=optimizer,
        vocab=vocab,
        device=device,
        num_epochs=args.epochs,
        start_epoch=start_epoch,
        checkpoint_path=str(checkpoint_path),
        plot_path=str(checkpoint_dir / "loss_curve.png"),
    )
    print(f"Done. Model: {checkpoint_path}   Vocabulary: {vocab_path}")


    main()


## Build the model

Create the custom encoder-decoder model, loss function, and optimizer. If a resume path is supplied, restore the saved model and optimizer state before continuing.


In [ ]:
# Build the custom CNN encoder-decoder and its training components.
model = CombinedModel(EMBED_SIZE, HIDDEN_SIZE, len(vocab), ATTENTION_DIM).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi["<PAD>"])
optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)

# Resume from a previous checkpoint only when one was selected above.
start_epoch = 0
if RESUME_FROM:
    last_epoch, last_loss = load_checkpoint(model, optimizer, RESUME_FROM)
    start_epoch = last_epoch + 1
    print(f"Resuming after epoch {last_epoch + 1} (loss {last_loss:.4f})")


## Start training

Run this cell only after setting `DATASET_PATH` and confirming the checkpoint output directory. It starts the existing training loop and writes checkpoints and the loss curve under `OUTPUT_DIR`.


In [ ]:
# Start training and write checkpoints plus the loss curve to the output folder.
losses = train_model(
    model=model,
    dataloader=dataloader,
    criterion=criterion,
    optimizer=optimizer,
    vocab=vocab,
    device=device,
    num_epochs=NUM_EPOCHS,
    start_epoch=start_epoch,
    checkpoint_path=CHECKPOINT_PATH,
    plot_path=os.path.join(OUTPUT_DIR, "loss_curve.png"),
)
